In [6]:
from sklearn.model_selection import GridSearchCV

print("GridSearchCV imported successfully!")

GridSearchCV imported successfully!


In [10]:
# Summary
# GridSearchCV automatically searches for the best
# XGBoost hyperparameters using 5-Fold Cross Validation.

import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    KFold,
    GridSearchCV
)

from xgboost import XGBRegressor


# Load Dataset

X = pd.read_csv(
    "../Outputs/pipeline1_train_preprocessed.csv"
)

y = np.log1p(

    pd.read_csv(
        "../Outputs/pipeline1_train_target.csv"
    )["SalePrice"]

)

print("Dataset Loaded Successfully!\n")

print(f"Dataset Shape : {X.shape}")

print(f"Missing Values: {X.isnull().sum().sum()}")


# Train / Test Split

X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.2,

    random_state=42

)


# Create 5-Fold Cross Validation

kf = KFold(

    n_splits=5,

    shuffle=True,

    random_state=42

)


# Parameter Grid

param_grid = {

    "n_estimators": [1000, 2000, 3000],

    "learning_rate": [0.01, 0.03, 0.05, 0.1],

    "max_depth": [3, 4, 5],

    "subsample": [0.7, 0.8, 1.0],

    "colsample_bytree": [0.7, 0.8, 1.0]

}


# Grid Search

grid_search = GridSearchCV(

    estimator=XGBRegressor(

        random_state=42,

        reg_alpha=0.1,

        reg_lambda=1.0,

        n_jobs=-1,

        verbosity=0

    ),

    param_grid=param_grid,

    scoring="neg_root_mean_squared_error",

    cv=kf,

    n_jobs=-1,

    verbose=3

)


# Train Grid Search

grid_search.fit(

    X_train.fillna(0),

    y_train

)


print("\nGrid Search Completed Successfully!\n")


print("=" * 60)

print("Best Parameters")

print("=" * 60)

print(grid_search.best_params_)


print()

print("=" * 60)

print("Best Cross Validation RMSE")

print("=" * 60)

print(f"{-grid_search.best_score_:.5f}")


# Best Model

best_xgb = grid_search.best_estimator_


# Test Performance

pred = best_xgb.predict(

    X_test.fillna(0)

)

rmse = np.sqrt(

    ((pred - y_test) ** 2).mean()

)


print()

print("=" * 60)

print("Test Performance")

print("=" * 60)

print(f"RMSE : {rmse:.5f}")

Dataset Loaded Successfully!

Dataset Shape : (1460, 208)
Missing Values: 1468
Fitting 5 folds for each of 324 candidates, totalling 1620 fits

Grid Search Completed Successfully!

Best Parameters
{'colsample_bytree': 0.8, 'learning_rate': 0.03, 'max_depth': 3, 'n_estimators': 1000, 'subsample': 0.8}

Best Cross Validation RMSE
0.11991

Test Performance
RMSE : 0.13152
